# From traffic demand to a scored network

Build a five-router network, one directed link at a time. The input is how much traffic each pair must exchange. Divi proposes which links to build. A routing solver sends the traffic through those links and measures the load on the busiest one.

Run the cells in order using SharQ’s `.venv` kernel. The default run uses the local Maestro simulator. To experiment, edit the **Build the QUBO** cell and rerun from there.

## Load the QOBLIB demand matrix

Rows are sources, columns are destinations, and router indices start at zero in Python. The helper reads the project’s cached copy and downloads the official file on first use if needed. Change `n_nodes` to select an instance from 5 to 24 routers. [QOBLIB demand data](https://github.com/ZIB-AOPT/QOBLIB/blob/main/08-network/instances/demand.txt)

In [43]:
from pathlib import Path

import qoblib_tools as qt

n_nodes = 11
if not 5 <= n_nodes <= 24:
    raise ValueError("Choose a QOBLIB instance between 5 and 24 routers.")

demand = qt.load_network_demand(n_nodes)
instance = f"network{n_nodes:02d}"
print(f"{instance} demand matrix (source rows, destination columns)")
print(demand)
print(f"Router 0 must send {demand[0, 4]} units to router 4.")

network11 demand matrix (source rows, destination columns)
[[ 0 24 43 23 21 41 61 21 20  0  0]
 [ 0  0  0  0 21 18 39 23  0  0 40]
 [16  0  0 20 44  0 42 22 20  0  0]
 [42 20 18  0 40 57 43 42  0  0 19]
 [ 0  0  0 40  0 83 60  0 44  0 37]
 [60 18  0 37 18  0 21 41 23 39  0]
 [22  0  0  0  0  0  0 61 36 80 96]
 [ 0  0 20 19 17 20 40  0  0 60 61]
 [21 22 38  0 44 20 40 39  0 36 22]
 [ 0 24 23 39 20  0  0 41  0  0  0]
 [60 57  0  0 16  0 16 37  0  0  0]]
Router 0 must send 21 units to router 4.


## Build the QUBO

One bit represents one possible directed link. For example, a 1 in the bit for `(0, 4)` builds the link from router 0 to router 4. Five routers give 20 bits, of which a valid network uses ten.

The dummy objective starts at zero. For each router, the degree penalty adds `(outgoing − 2)² + (incoming − 2)²`. Every degree-valid layout therefore has the same energy. For a first experiment, set `direct_demand_weight = 1.0` to reward direct links in proportion to their demand. The routing cell will tell you whether that change reduces congestion.

In [44]:
direct_demand_weight = 0.0
degree_weight = 2.0

# The helper returns two separate binary models and the bit-to-link mapping.
direct_demand, degree_penalty, arcs = qt.build_network_qubo(demand)
objective = direct_demand * direct_demand_weight
link_bit = {arc: bit for bit, arc in enumerate(arcs)}

# Edit the objective here. For example, uncomment this to reward a two-hop path.
# objective.add_quadratic(link_bit[0, 1], link_bit[1, 4], -0.5)

qubo = objective + degree_weight * degree_penalty
Q, offset = qubo.to_qubo()  # Sparse coefficients. A diagonal entry weights a single bit.
bit = link_bit[0, 4]
print(f"{len(arcs)} variables; bit {bit} builds link 0 → 4")
print(f"Example Q coefficient: Q[{bit}, {bit}] = {Q[bit, bit]:g}; offset = {offset:g}")

110 variables; bit 3 builds link 0 → 4
Example Q coefficient: Q[3, 3] = -12; offset = 176


## Run partitioned QAOA in Divi

Divi converts the binary model to an Ising cost Hamiltonian. The community decomposer groups link variables, QAOA samples each group, and the ensemble combines the partial answers. The cluster limit counts **link variables**. The default of eight makes even the five-router example use partitioning.

Leave `use_cloud` off for a local run. To submit circuits to Qoro’s cloud, configure `QORO_API_KEY`, set `use_cloud = True`, and run this cell. A cloud run uses account credits.

In [45]:
import hybrid
from divi.backends import JobConfig, MaestroSimulator, QoroService
from divi.qprog.ensembles import BeamSearchStrategy, PartitioningProgramEnsemble
from divi.qprog.optimizers import MonteCarloOptimizer
from divi.qprog.problems import BinaryOptimizationProblem, CommunityDecomposer, QUBOPartitioningConfig

use_cloud = True
shots = 2_000
cluster_size = 8

if use_cloud:
    backend = QoroService(job_config=JobConfig(shots=shots, simulator_cluster="qoro_maestro"))
else:
    backend = MaestroSimulator(shots=shots)

problem = BinaryOptimizationProblem(
    qubo,
    decomposer=CommunityDecomposer(QUBOPartitioningConfig(max_n_variables_per_cluster=cluster_size)),
    composer=hybrid.SplatComposer(),
)
ensemble = PartitioningProgramEnsemble(
    problem=problem,
    quantum_routine="qaoa",
    n_layers=1,
    optimizer=MonteCarloOptimizer(population_size=8),
    max_iterations=3,
    backend=backend,
    reporting_level="off",
    seed=7,
)
ensemble.run()
candidates = ensemble.get_top_solutions(
    n=24, strategy=BeamSearchStrategy(beam_width=24, n_partition_candidates=8)
)
print(f"{type(backend).__name__} returned {len(candidates)} candidate layouts.")

QoroService returned 24 candidate layouts.


## Decode the candidates and evaluate congestion

Turn each combined bit assignment back into a list of directed links. If its degrees are wrong, the repair step keeps as many sampled links as possible while restoring two incoming and two outgoing links per router. It uses direct demand to break ties between equally close repairs. The routing solver can split traffic over paths. It gives an infinite score to a layout that cannot deliver all demand.

Score each distinct repaired layout once. Keep its score for the swap search and display the best starting layout below. Keep its original bits so you can inspect exactly what repair changed. Layouts are scored in thousandths of a traffic unit.

In [46]:
import math

layout_scores = {}
best_score = math.inf
for bits, _ in candidates:
    sample = dict(zip(qubo.variables, bits, strict=True))
    raw_arcs = qt.decode_topology(sample, arcs)
    if qt.is_valid_topology(n_nodes, raw_arcs):
        topology = raw_arcs
    else:
        topology = qt.repair_topology(n_nodes, raw_arcs, demand)

    layout = tuple(sorted(topology))
    if layout in layout_scores:
        continue
    score = qt.route_min_congestion(demand, layout).objective
    layout_scores[layout] = score
    if score < best_score:
        best_score = score
        best_raw = set(raw_arcs)
        best_arcs = sorted(topology)
        best_bits = "".join(str(int(sample[v])) for v in range(len(arcs)))

if not math.isfinite(best_score):
    raise RuntimeError("No routable candidate. Increase the beam width and candidate count, then rerun.")

print("Divi bits in link order:", best_bits)
print("Decoded links:", sorted(best_raw))
print("Removed by repair:", sorted(best_raw - set(best_arcs)))
print("Added by repair:", sorted(set(best_arcs) - best_raw))
print("Evaluated links:", best_arcs)
print(f"Routing LP score (rounded up): {best_score:,.0f}")

Divi bits in link order: 00100001011100100000000000010100000111001000100010000110001001000100010000100010001001011000110010001101000000
Decoded links: [(0, 3), (0, 8), (0, 10), (1, 0), (1, 2), (1, 5), (2, 8), (2, 10), (3, 6), (3, 7), (3, 8), (4, 0), (4, 5), (4, 9), (5, 3), (5, 4), (5, 9), (6, 1), (6, 5), (6, 10), (7, 4), (7, 9), (8, 2), (8, 5), (8, 7), (8, 9), (9, 2), (9, 3), (9, 6), (10, 0), (10, 1), (10, 3)]
Removed by repair: [(0, 10), (1, 0), (3, 8), (4, 9), (5, 3), (6, 5), (8, 5), (8, 9), (9, 2), (10, 3)]
Added by repair: []
Evaluated links: [(0, 3), (0, 8), (1, 2), (1, 5), (2, 8), (2, 10), (3, 6), (3, 7), (4, 0), (4, 5), (5, 4), (5, 9), (6, 1), (6, 10), (7, 4), (7, 9), (8, 2), (8, 7), (9, 3), (9, 6), (10, 0), (10, 1)]
Routing LP score (rounded up): 304,200


## Refine the top five layouts

Starting from each of the five lowest routing scores, try 200 legal link swaps. Swapping the destinations of two links preserves all incoming and outgoing degrees. Skip self-links and duplicate links. Keep a swap only if routing the traffic gives a lower score.

This cell runs locally on the candidates from the decode cell. Each row shows one starting score, how many swaps improved it, and the checked integer score at the end. Fewer rows means fewer than five distinct routable candidates were available.

In [47]:
from random import Random
from tempfile import TemporaryDirectory

n_starts = 5
swaps_per_start = 200
seed = 7

def refine_layout(layout, start_score, rng):
    edges = set(layout)
    score = start_score
    tried = accepted = 0
    while tried < swaps_per_start:
        first, second = rng.sample(sorted(edges), 2)
        (a, b), (c, d) = first, second
        replacements = {(a, d), (c, b)}
        if a == d or c == b or replacements & edges:
            continue

        trial = (edges - {first, second}) | replacements
        trial_score = qt.route_min_congestion(demand, trial).objective
        tried += 1
        if trial_score < score:
            edges, score = trial, trial_score
            accepted += 1
    return sorted(edges), accepted

starts = sorted((score, layout) for layout, score in layout_scores.items() if math.isfinite(score))
best_checked_score = math.inf
print("Start   Initial LP   Accepted   Checked score")
with TemporaryDirectory() as scratch:
    trial_file = Path(scratch) / f"{instance}.sol"
    for rank, (start_score, layout) in enumerate(starts[:n_starts], start=1):
        refined, accepted = refine_layout(layout, start_score, Random(seed + rank - 1))
        candidate_routing = qt.route_integral(demand, refined)
        qt.write_network_solution(trial_file, n_nodes, candidate_routing)
        valid, score, message = qt.check_network_solution(trial_file, n_nodes, demand)
        if not valid:
            raise ValueError(f"Start {rank} failed the solution checker: {message}")
        print(f"{rank:>5}   {start_score:>10,.0f}   {accepted:>8}   {score:>13,}")
        if score < best_checked_score:
            best_checked_score, best_routing = score, candidate_routing

print(f"Lowest checked score: {best_checked_score:,}")

Start   Initial LP   Accepted   Checked score
    1      304,200          8         252,000
    2      307,250         16         264,834
    3      313,600         13         263,834
    4      321,667         11         256,667
    5      325,250         10         274,334
Lowest checked score: 252,000


## Save the best checked network

Write the lowest-scoring finalist to `starter_results/multistart/networkNN.sol`. The checker reads the saved file again and the final lines identify the busiest links. Rerunning this cell replaces the multistart output for this instance.

In [48]:
routing = best_routing
result_path = Path("starter_results/multistart") / f"{instance}.sol"
result_path.parent.mkdir(parents=True, exist_ok=True)
qt.write_network_solution(result_path, n_nodes, routing)
valid, checked_score, message = qt.check_network_solution(result_path, n_nodes, demand)
if not valid:
    raise ValueError(f"QOBLIB checker rejected the solution: {message}")

link_load = dict.fromkeys(routing.arcs, 0)
for (source, i, j), flow in routing.flows.items():
    link_load[i, j] += flow
busiest = [arc for arc, load in link_load.items() if load == checked_score]
print(f"Checked solution: {result_path} ({message})")
print(f"Busiest links: {busiest}; peak load: {checked_score / qt.INTSCALE:,.3f} traffic units")
print(f"QOBLIB score: {checked_score:,}")
if n_nodes == 5:
    print(f"Five-router optimum: 65,500; gap: {(checked_score / 65_500 - 1):.1%}")

Checked solution: starter_results/multistart/network11.sol (valid)
Busiest links: [(0, 6), (1, 7), (2, 6), (4, 0), (4, 8), (5, 9), (8, 1), (8, 5)]; peak load: 252.000 traffic units
QOBLIB score: 252,000
